In [1]:
import copy
import logging
import warnings

import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms

warnings.filterwarnings("ignore")


In [2]:
# Logging 설정
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(name)s - %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
)
logger = logging.getLogger(__name__)


In [ ]:
# Training hyperparameters
BATCH_SIZE = 32
LEARNING_RATE = 0.001
NUM_EPOCHS = 10
PATIENCE = 3  
VALID_RATIO = 0.1
RANDOM_SEED = 42
DATA_ROOT = "./data"

# Model architecture config
IN_CHANNELS = 1
NUM_CLASSES = 10
CONV1_OUT = 32
CONV2_OUT = 64
KERNEL_SIZE = 3
PADDING = 1
POOL_SIZE = 2
FC_HIDDEN = 128
IMAGE_SIZE = 28
DROPOUT_CONV_P = 0.25
DROPOUT_FC_P = 0.5


In [4]:
def load_datasets(root=DATA_ROOT, download=True):
    """
    MNIST train/test 데이터셋을 로드하는 함수
    :param root: 데이터 저장 경로
    :param download: 데이터가 없을 때 다운로드 여부
    :return: train_set, test_set
    """
    transform = transforms.Compose(
        [transforms.ToTensor(), transforms.Normalize((0.5,), (0.5,))]
    )

    train_set = torchvision.datasets.MNIST(
        root=root, train=True, download=download, transform=transform
    )
    test_set = torchvision.datasets.MNIST(
        root=root, train=False, download=download, transform=transform
    )

    return train_set, test_set


In [5]:
def split_train_valid(train_set, valid_ratio=VALID_RATIO, seed=RANDOM_SEED):
    """
    train_set을 train/valid subset으로 분리하는 함수
    :param train_set: 전체 훈련 데이터셋
    :param valid_ratio: validation 비율
    :param seed: 재현성을 위한 random seed
    :return: train_subset, valid_subset
    """
    train_size = int(len(train_set) * (1 - valid_ratio))
    valid_size = len(train_set) - train_size

    train_subset, valid_subset = torch.utils.data.random_split(
        train_set,
        [train_size, valid_size],
        generator=torch.Generator().manual_seed(seed),
    )

    return train_subset, valid_subset


In [6]:
def create_dataloaders(train_subset, valid_subset, test_set, batch_size=BATCH_SIZE):
    """
    데이터셋을 DataLoader로 변환하는 함수
    :param train_subset: 훈련용 subset
    :param valid_subset: 검증용 subset
    :param test_set: 테스트 데이터셋
    :param batch_size: 배치 크기
    :return: train_loader, valid_loader, test_loader
    """
    train_loader = torch.utils.data.DataLoader(
        train_subset, batch_size=batch_size, shuffle=True
    )
    valid_loader = torch.utils.data.DataLoader(
        valid_subset, batch_size=batch_size, shuffle=False
    )
    test_loader = torch.utils.data.DataLoader(
        test_set, batch_size=batch_size, shuffle=False
    )

    return train_loader, valid_loader, test_loader


In [7]:
def explore_labels(train_set, test_set):
    """
    라벨 종류와 분포를 로깅하는 함수 (데이터 검증 목적)
    - dataset.targets 를 직접 사용 (전체 반복 인덱싱 대비 빠름)
    :param train_set: 훈련 데이터셋
    :param test_set: 테스트 데이터셋
    """
    logger.info("train_set size: %d", len(train_set))
    logger.info("test_set size: %d", len(test_set))

    unique_labels_train = sorted(set(train_set.targets.tolist()))
    unique_labels_test = sorted(set(test_set.targets.tolist()))
    logger.info("Unique labels (train): %s", unique_labels_train)
    logger.info("Unique labels (test): %s", unique_labels_test)

    logger.info(
        "Label distribution (train): %s", torch.bincount(train_set.targets).tolist()
    )
    logger.info(
        "Label distribution (test): %s", torch.bincount(test_set.targets).tolist()
    )


In [8]:
class CNN(nn.Module):
    """
    CNN Architecture 정의 클래스
    - Conv-BatchNorm-ReLU-Pool 블록 x2 + Dropout + FC-BatchNorm-ReLU-Dropout + FC
    - 채널 수, 커널, 클래스 수 등은 모듈 상수를 기본값으로 하되
      생성자 인자로도 주입할 수 있도록 구성
    """

    def __init__(
        self,
        in_channels=IN_CHANNELS,
        num_classes=NUM_CLASSES,
        conv1_out=CONV1_OUT,
        conv2_out=CONV2_OUT,
        kernel_size=KERNEL_SIZE,
        padding=PADDING,
        pool_size=POOL_SIZE,
        fc_hidden=FC_HIDDEN,
        image_size=IMAGE_SIZE,
        dropout_conv_p=DROPOUT_CONV_P,
        dropout_fc_p=DROPOUT_FC_P,
    ):
        super(CNN, self).__init__()
        self.conv1 = nn.Conv2d(
            in_channels, conv1_out, kernel_size=kernel_size, padding=padding
        )
        self.bn1 = nn.BatchNorm2d(conv1_out)
        self.conv2 = nn.Conv2d(
            conv1_out, conv2_out, kernel_size=kernel_size, padding=padding
        )
        self.bn2 = nn.BatchNorm2d(conv2_out)
        self.pool = nn.MaxPool2d(kernel_size=pool_size, stride=pool_size)
        self.dropout_conv = nn.Dropout(dropout_conv_p)

        # conv+pool 을 2번 통과한 뒤의 feature map 한 변 크기 (하드코딩 제거)
        feature_size = image_size // (pool_size ** 2)
        self.fc1 = nn.Linear(conv2_out * feature_size * feature_size, fc_hidden)
        self.bn_fc = nn.BatchNorm1d(fc_hidden)
        self.dropout_fc = nn.Dropout(dropout_fc_p)
        self.fc2 = nn.Linear(fc_hidden, num_classes)

    def forward(self, x):
        """
        Forward 정의
        :param x: 입력 이미지 텐서
        :return: 모델 출력값
        """
        x = self.pool(torch.relu(self.bn1(self.conv1(x))))
        x = self.pool(torch.relu(self.bn2(self.conv2(x))))
        x = self.dropout_conv(x)

        x = torch.flatten(x, 1)
        x = torch.relu(self.bn_fc(self.fc1(x)))
        x = self.dropout_fc(x)
        x = self.fc2(x)

        return x


In [9]:
def train_model(
    model,
    train_loader,
    valid_loader,
    device,
    criterion,
    optimizer,
    num_epochs=NUM_EPOCHS,
    patience=PATIENCE,
):
    """
    CNN 모델 학습 함수 (validation 및 early stopping 포함)
    :param model: 학습할 모델
    :param train_loader: 훈련 데이터 로더
    :param valid_loader: 검증 데이터 로더
    :param device: CPU or GPU
    :param criterion: 손실 함수
    :param optimizer: 최적화 함수
    :param num_epochs: 최대 에폭 수
    :param patience: valid_loss가 개선되지 않아도 허용하는 연속 epoch 수
    :return: train_losses, valid_losses, valid_accuracies (epoch 별 기록)
    """
    train_losses = []
    valid_losses = []
    valid_accuracies = []

    best_valid_loss = float("inf")
    best_model_state = None
    patience_counter = 0

    for epoch in range(num_epochs):
        # ---- Training ----
        model.train()
        running_loss = 0.0

        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()

        train_loss = running_loss / len(train_loader)
        train_losses.append(train_loss)

        # ---- Validation ----
        model.eval()
        valid_running_loss = 0.0
        valid_correct = 0
        valid_total = 0

        with torch.no_grad():
            for inputs, labels in valid_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, labels)

                valid_running_loss += loss.item()

                _, predicted = torch.max(outputs, 1)
                valid_total += labels.size(0)
                valid_correct += (predicted == labels).sum().item()

        valid_loss = valid_running_loss / len(valid_loader)
        valid_accuracy = valid_correct / valid_total
        valid_losses.append(valid_loss)
        valid_accuracies.append(valid_accuracy)

        logger.info(
            "Epoch [%d/%d] - train_loss: %.4f, valid_loss: %.4f, valid_accuracy: %.4f",
            epoch + 1,
            num_epochs,
            train_loss,
            valid_loss,
            valid_accuracy,
        )

        # ---- Early Stopping ----
        if valid_loss < best_valid_loss:
            best_valid_loss = valid_loss
            best_model_state = copy.deepcopy(model.state_dict())
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                logger.info(
                    "Early stopping triggered at epoch %d (best valid_loss: %.4f)",
                    epoch + 1,
                    best_valid_loss,
                )
                break

    if best_model_state is not None:
        model.load_state_dict(best_model_state)
        logger.info(
            "Restored model weights from best valid_loss: %.4f", best_valid_loss
        )

    return train_losses, valid_losses, valid_accuracies


In [10]:
def evaluate_model(model, loader, device):
    """
    CNN 평가 함수 (accuracy 계산)
    :param model: 평가할 모델
    :param loader: 평가할 데이터 로더
    :param device: CPU or GPU
    :return: accuracy
    """
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for inputs, labels in loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, predicted = torch.max(outputs, 1)

            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    accuracy = correct / total

    return accuracy


In [11]:
def evaluate_label_wise(model, loader, device, num_classes=NUM_CLASSES):
    """
    라벨(클래스)별 accuracy를 계산하고 로깅하는 함수
    - 특정 라벨에서만 성능이 낮은지(모델 편향, 데이터 편중 등) 모니터링 목적
    :param model: 평가할 모델
    :param loader: 평가할 데이터 로더
    :param device: CPU or GPU
    :param num_classes: 클래스(라벨) 개수
    :return: 라벨별 accuracy 리스트
    """
    model.eval()
    label_correct = torch.zeros(num_classes)
    label_total = torch.zeros(num_classes)

    with torch.no_grad():
        for inputs, labels in loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, predicted = torch.max(outputs, 1)

            for label in range(num_classes):
                mask = labels == label
                label_total[label] += mask.sum().item()
                label_correct[label] += (predicted[mask] == labels[mask]).sum().item()

    label_accuracy = (label_correct / label_total).tolist()

    for label in range(num_classes):
        logger.info(
            "Label %d accuracy: %d/%d (%.4f)",
            label,
            int(label_correct[label]),
            int(label_total[label]),
            label_accuracy[label],
        )

    return label_accuracy


In [12]:
def main():
    """
    전체 실행 흐름을 제어하는 메인 함수
    """
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    logger.info("Using device: %s", device)

    # 1) 데이터 로딩 & 검증
    train_set, test_set = load_datasets(download=True)
    explore_labels(train_set, test_set)

    train_subset, valid_subset = split_train_valid(
        train_set, VALID_RATIO, RANDOM_SEED
    )
    logger.info(
        "Split - train: %d, valid: %d, test: %d",
        len(train_subset),
        len(valid_subset),
        len(test_set),
    )

    train_loader, valid_loader, test_loader = create_dataloaders(
        train_subset, valid_subset, test_set, BATCH_SIZE
    )

    # 2) 모델 구성
    model = CNN().to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

    # 3) 학습 (validation + early stopping 포함)
    train_model(
        model,
        train_loader,
        valid_loader,
        device,
        criterion,
        optimizer,
        num_epochs=NUM_EPOCHS,
        patience=PATIENCE,
    )

    # 4) 평가 (train/valid/test accuracy)
    train_accuracy = evaluate_model(model, train_loader, device)
    valid_accuracy = evaluate_model(model, valid_loader, device)
    test_accuracy = evaluate_model(model, test_loader, device)

    logger.info("Train accuracy: %.4f", train_accuracy)
    logger.info("Valid accuracy: %.4f", valid_accuracy)
    logger.info("Test accuracy: %.4f", test_accuracy)

    # 5) 라벨별 accuracy 모니터링 (test set 기준)
    evaluate_label_wise(model, test_loader, device, NUM_CLASSES)


In [13]:
# 실행
if __name__ == "__main__":
    main()


2026-08-03 10:08:12 [INFO] __main__ - Using device: cpu
2026-08-03 10:08:12 [INFO] __main__ - train_set size: 60000
2026-08-03 10:08:12 [INFO] __main__ - test_set size: 10000
2026-08-03 10:08:12 [INFO] __main__ - Unique labels (train): [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
2026-08-03 10:08:12 [INFO] __main__ - Unique labels (test): [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
2026-08-03 10:08:12 [INFO] __main__ - Label distribution (train): [5923, 6742, 5958, 6131, 5842, 5421, 5918, 6265, 5851, 5949]
2026-08-03 10:08:12 [INFO] __main__ - Label distribution (test): [980, 1135, 1032, 1010, 982, 892, 958, 1028, 974, 1009]
2026-08-03 10:08:12 [INFO] __main__ - Split - train: 54000, valid: 6000, test: 10000
2026-08-03 10:08:27 [INFO] __main__ - Epoch [1/10] - train_loss: 0.1757, valid_loss: 0.0539, valid_accuracy: 0.9840
2026-08-03 10:08:42 [INFO] __main__ - Epoch [2/10] - train_loss: 0.0833, valid_loss: 0.0367, valid_accuracy: 0.9888
2026-08-03 10:08:56 [INFO] __main__ - Epoch [3/10] - train_loss: 0.0689, vali

---
End of Documents